In [ ]:
# Este cuaderno se encarga de:
# 1. Descargar o cargar desde cache el dataset OSV-5M
# 2. Convierte las coordenadas de cada imagen a un "Token S2" del nivel elegido
#    S2 es un sistema de Google que divide la esfera terrestre en celdas de tamaño variable
# 3. Filtrado de clases, las celdas que tengan menos imágenes que el mínimo se eliminan
# 4. Genera un diccionario ("s2_class_map.pkl") con la información de cada celda válida
# 5. Genera un mapa visual ("mapa_cobertura.html") con los centros de las celdas para poder visualizarlos

import os
import sys
import multiprocessing

CACHE_DIR = "D:/Datasets_Cache"                                 # Ruta donde guardar el dataset
MODELS_DIR = "../models"                                        # Carpeta donde guardar el output del notebook
OUTPUT_FILE = os.path.join(MODELS_DIR,"s2_class_map.pkl")       # Archivo final de clases
MAP_FILENAME = os.path.join(MODELS_DIR,"mapa_cobertura.html")   # Archivo visual con los centros de las celdas
S2_LEVEL = 12                                                   # Nivel de precisión o tamaño de celda (~3km cuadrados con nivel = 12)
MIN_IMAGES_PER_CLASS = 3                                        # Mínimo de fotos para aceptar una celda
NUM_CORES = max(1, multiprocessing.cpu_count() - 1)             # Dejar al menos 1 libre para el SO

# Forzamos la caché en el disco grande ANTES de importar datasets
os.environ["HF_DATASETS_CACHE"] = CACHE_DIR
os.makedirs(CACHE_DIR, exist_ok=True)

# Instalación automática de dependencias
print("Verificando librerías...")
os.system('pip install -q "datasets<3.0.0" s2sphere folium')
print(f"Configuración lista. Usando caché en: {CACHE_DIR}")
print(f"Núcleos disponibles para procesamiento: {NUM_CORES}")

In [ ]:
import s2sphere
import collections
import pickle
import random
import folium

# Esta función se encarga de convertir coordenadas a tokens S2 
# Optimizada para ejecutarse en paralelo (por eso el import local)
def batch_process_s2(batch):
    import s2sphere
    
    # Constante local para los workers
    LEVEL = 12 
    
    tokens = []
    lats = batch['latitude']
    lons = batch['longitude']
    
    for lat, lon in zip(lats, lons):
        token = None 
        try:
            # Validación de coordenadas
            if lat is not None and lon is not None:
                # Evitar el (0,0) y rangos imposibles
                if not (abs(lat) > 90 or abs(lon) > 180 or (lat == 0 and lon == 0)):
                    p1 = s2sphere.LatLng.from_degrees(lat, lon)
                    token = s2sphere.CellId.from_lat_lng(p1).parent(LEVEL).to_token()
        except Exception:
            pass # Si falla, se mantiene como None
            
        tokens.append(token)
            
    return {"s2_token": tokens}

# Función para validar la configuración de S2, si falla se detiene el proceso
def validate_s2_setup(dataset):
    print("\nRealizando prueba de diagnóstico...")
    try:
        sample = dataset[0]
        lat, lon = sample['latitude'], sample['longitude']
        p1 = s2sphere.LatLng.from_degrees(lat, lon)
        token = s2sphere.CellId.from_lat_lng(p1).parent(12).to_token()
        print(f"Prueba exitosa. Lat: {lat:.4f}, Lon: {lon:.4f} -> Token: {token}")
        return True
    except Exception as e:
        print(f"\nError crítico al validar S2: {e}")
        return False

In [ ]:
import time
from datasets import load_dataset

def run_etl():
    print(f"Cargando OSV-5M desde caché local...")
    
    try:
        # Carga optimizada sin streaming (en local)
        dataset = load_dataset(
            "osv5m/osv5m", 
            split="train", 
            streaming=False,                # Modo de carga, estando en false primero descarga todo el dataset y luego empieza a trabajar
            trust_remote_code=True,         # Autoriza a cargar el dataset con el script de carga de Hugging Face
            keep_in_memory=False            # No mantiene todo el dataset en memoria ram (para evitar que colapse)
        )
        
        # Mantenemos solo las columnas de latitud y longitud, ahorrando memoria
        cols_to_keep = {'latitude', 'longitude'}
        cols_to_drop = [c for c in dataset.column_names if c not in cols_to_keep]
        dataset = dataset.remove_columns(cols_to_drop)
        print(f"Dataset listo: {len(dataset):,} filas.")
        
    except Exception as e:
        print(f"Error cargando dataset: {e}")
        return None

    # Verificación antes de empezar a procesar
    if not validate_s2_setup(dataset): return

    print(f"\nIniciando procesamiento en paralelo con {NUM_CORES} núcleos...")
    start_time = time.time()

    results = dataset.map(
        batch_process_s2,
        batched=True,
        batch_size=5000,                            # Cantidad de coordenadas por batch
        num_proc=NUM_CORES,
        remove_columns=['latitude', 'longitude'],   # Una vez calculado el token, no las necesitamos
        desc="Generando Tokens S2"
    )

    print("Agregando frecuencias...")
    # Filtramos los None
    all_tokens = [t for t in results['s2_token'] if t is not None]
    
    counter = collections.Counter(all_tokens)
    elapsed = time.time() - start_time
    print(f"Procesado en {elapsed:.2f} segundos.")

    print("Guardando archivo de clases...")
    valid_cells = [cell for cell, count in counter.items() if count >= MIN_IMAGES_PER_CLASS]
    valid_cells.sort(key=lambda x: counter[x], reverse=True)
    
    # Índices para la IA (Token -> ID numérico y viceversa)
    cell_to_idx = {cell: idx for idx, cell in enumerate(valid_cells)}
    idx_to_cell = {idx: cell for cell, idx in cell_to_idx.items()}
    
    final_data = {
        "cell_to_idx": cell_to_idx,
        "idx_to_cell": idx_to_cell,
        "total_cells": len(valid_cells),
        "s2_level": S2_LEVEL,
        "raw_counts": dict(counter)
    }
    
    with open(OUTPUT_FILE, "wb") as f:
        pickle.dump(final_data, f)
        
    print(f"Clases generadas: {len(valid_cells):,}")
    print(f"Archivo guardado: {os.path.abspath(OUTPUT_FILE)}")
    return OUTPUT_FILE

# Ejecutar
output_pkl = run_etl()

In [ ]:
def generate_map(pkl_file, html_file, samples=400000):
    if not pkl_file or not os.path.exists(pkl_file):
        print("No hay archivo PKL para visualizar")
        return

    print(f"\nGenerando mapa interactivo ({samples} puntos)")
    with open(pkl_file, "rb") as f:
        data = pickle.load(f)
        
    all_cells = list(data["cell_to_idx"].keys())
    
    # Muestreo aleatorio si hay demasiados puntos
    sampled_cells = random.sample(all_cells, min(samples, len(all_cells)))
    
    # Mapa base oscuro
    m = folium.Map(location=[20, 0], zoom_start=2, tiles='CartoDB dark_matter')
    
    count = 0
    for token in sampled_cells:
        try:
            cell_id = s2sphere.CellId.from_token(token)
            ll = cell_id.to_lat_lng()
            folium.CircleMarker(
                location=[ll.lat().degrees, ll.lng().degrees],
                radius=1,
                color='#33FF57', # Verde Matrix
                fill=True,
                fill_opacity=0.6,
                weight=0
            ).add_to(m)
            count += 1
        except:
            continue
            
    m.save(html_file)
    print(f"Mapa guardado en: {os.path.abspath(html_file)}")
    print("Ábrelo en tu navegador para ver la cobertura mundial.")

# Ejecutar
generate_map(OUTPUT_FILE, MAP_FILENAME)